# Logistic Regression

This notebook shows binary logistic regression using the package implementation. The math is explained in markdown and tiny formula cells; training uses `LogisticRegressionGD`.

Author: Pasquale Marzaioli

## Model Formula

Logistic regression first computes a linear score:

```text
z = X @ w + b
```

Then it maps the score to a probability:

```text
p = sigmoid(z) = 1 / (1 + exp(-z))
```

A probability `p >= 0.5` becomes class `1`; otherwise it becomes class `0`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from ml_from_scratch import LogisticRegressionGD
from ml_from_scratch.datasets import make_binary_classification_data
from ml_from_scratch.logistic_regression import binary_cross_entropy_loss, sigmoid
from ml_from_scratch.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from ml_from_scratch.plotting import (
    plot_binary_cross_entropy_curve,
    plot_logistic_regression_fit,
)
from ml_from_scratch.preprocessing import (
    normalize_features,
    train_test_split,
    transform_features,
)

## A Tiny Formula Check

This cell computes sigmoid probabilities directly for a few scores.

In [ ]:
scores_demo = np.array([-2.0, 0.0, 2.0])
probabilities_demo = sigmoid(scores_demo)

print(np.round(probabilities_demo, 3))

## Generate And Split Data

The classification helper creates labels by thresholding noisy linear scores at zero.

In [ ]:
X, y = make_binary_classification_data(
    n_samples=120,
    weights=np.array([1.0]),
    noise=0.6,
    random_state=19,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=19,
)

print(X_train.shape, X_test.shape)
print(f"class counts: {np.bincount(y)}")

## Normalize Features

The same training means and scales are reused for test data and plotting grids.

In [ ]:
X_train_normalized, means, scales = normalize_features(X_train)
X_test_normalized = transform_features(X_test, means, scales)

print(f"training mean after normalization: {np.mean(X_train_normalized):.3f}")
print(f"training std after normalization: {np.std(X_train_normalized):.3f}")

## Train The Model

Binary cross-entropy is low when predicted probabilities are close to the true labels.

In [ ]:
model = LogisticRegressionGD(learning_rate=0.2, n_iterations=1000)
model.fit(X_train_normalized, y_train)

print(f"first BCE: {model.loss_history_[0]:.3f}")
print(f"final BCE: {model.loss_history_[-1]:.3f}")

In [ ]:
ax = plot_binary_cross_entropy_curve(model.loss_history_)
plt.show()

In [ ]:
x_grid = np.linspace(float(np.min(X)), float(np.max(X)), 200).reshape(-1, 1)
x_grid_normalized = transform_features(x_grid, means, scales)
probability_grid = model.predict_proba(x_grid_normalized)

ax = plot_logistic_regression_fit(X_train, y_train, x_grid, probability_grid)
plt.show()

## Evaluate Classification Metrics

Accuracy measures all correct labels. Precision, recall, and F1 focus on class `1`, the positive class.

In [ ]:
test_probabilities = model.predict_proba(X_test_normalized)
test_predictions = model.predict(X_test_normalized)

print(f"test BCE: {binary_cross_entropy_loss(y_test, test_probabilities):.3f}")
print(f"accuracy: {accuracy_score(y_test, test_predictions):.3f}")
print(f"precision: {precision_score(y_test, test_predictions):.3f}")
print(f"recall: {recall_score(y_test, test_predictions):.3f}")
print(f"F1: {f1_score(y_test, test_predictions):.3f}")